# Introduction to Parallelization in Python

Have you ever had a Python script that took hours to run, while your computer's fans stayed perfectly silent? By default, Python runs on a **single CPU core** due to the Global Interpreter Lock (GIL).

**Parallelization** is the process of breaking a large problem into smaller sub-tasks that can be executed simultaneously across **multiple CPU cores**.

### Why use parallelization?
* **Speed:** Reduce execution time drastically for heavy computations.
* **Efficiency:** Fully utilize your modern multi-core processor.

### This exercise's goal
We will simulate a heavy array-processing task (like image processing or data transformation) and compare a traditional sequential loop against a parallelized approach.

In [1]:
import numpy as np
import time
import math
from multiprocessing import cpu_count
from concurrent.futures import ProcessPoolExecutor

# Verify how many CPU cores we have available
cores = cpu_count()
print(f"Your machine has {cores} CPU cores available for parallelization.")

Your machine has 22 CPU cores available for parallelization.


## 1. The Heavy Array Operation

Let's create a CPU-intensive function. Imagine we have a large array of numbers, and for each number, we need to perform a series of complex mathematical operations (such as calculating trends, factorials,  or trigonometry).

We will use standard Python math operations inside the function to simulate a heavy workload that prevents automatic optimization.

In [2]:
# Generate a dataset of 5,000,000 numbers split into 50 large chunks
# Processing chunks allows us to distribute chunks to different cores
num_chunks = 50
chunk_size = 100_000
data_chunks = [np.random.uniform(1, 100, chunk_size) for _ in range(num_chunks)]

def heavy_array_operation(array):
    """
    Simulates a heavy, CPU-bound operation on an array of data.
    """
    output = []
    for x in array:
        # A series of math operations to force the CPU to work hard
        val = np.sin(x) * np.cos(x)
        val = np.exp(np.log(np.abs(val) + 1))
        output.append(val)
    return np.array(output)

print(f"Generated {num_chunks} chunks. Total elements to process: {num_chunks * chunk_size:,}")

Generated 50 chunks. Total elements to process: 5,000,000


In [3]:
len(data_chunks)

50

## 2. The Sequential Approach (Single Core)

First, let's process these chunks the traditional way: one after the other using a standard `for` loop. This utilizes exactly **one** CPU core.

In [4]:
print("Starting sequential processing...")
start_time = time.time()

sequential_results = []
for chunk in data_chunks:
    result = heavy_array_operation(chunk)
    sequential_results.append(result)

sequential_time = time.time() - start_time
print(f"Sequential processing finished in: {sequential_time:.2f} seconds")

Starting sequential processing...
Sequential processing finished in: 10.49 seconds


## 3. The Parallel Approach (Multi-Core)

Now, we will use Python's `concurrent.futures.ProcessPoolExecutor` (see https://docs.python.org/3/library/concurrent.futures.html). This automatically splits our chunks across our available CPU cores.

If you have a 4-core machine, it can theoretically process 4 chunks at the exact same time.

In [5]:
print(f"Starting parallel processing using {cores} cores...")
start_time = time.time()

# ProcessPoolExecutor handles the creation and management of child processes
with ProcessPoolExecutor(max_workers=cores) as executor:
    # executor.map automatically applies the function to each chunk in parallel
    parallel_results = list(executor.map(heavy_array_operation, data_chunks))

parallel_time = time.time() - start_time
print(f"Parallel processing finished in: {parallel_time:.2f} seconds")

Starting parallel processing using 22 cores...
Parallel processing finished in: 1.36 seconds


## 4. Performance Comparison

Let's sanity-check the results to make sure both methods yielded the exact same data, and calculate our speedup factor!

In [8]:
# Verify accuracy
assert len(sequential_results) == len(parallel_results), "Result lengths don't match!"
print("Verification passed: Both methods produced identical results lengths.")

# Calculate speedup
speedup = sequential_time / parallel_time
print("\n--- PERFORMANCE SUMMARY ---")
print(f"Sequential Time: {sequential_time:.2f} seconds")
print(f"Parallel Time:   {parallel_time:.2f} seconds")
print(f"Speedup Factor:  {speedup:.2f}x faster")

# Visual representation
efficiency = (speedup / cores) * 100
print(f"Core Utilization Efficiency: {efficiency:.1f}%")

Verification passed: Both methods produced identical results lengths.

--- PERFORMANCE SUMMARY ---
Sequential Time: 11.10 seconds
Parallel Time:   2.11 seconds
Speedup Factor:  5.26x faster
Core Utilization Efficiency: 23.9%


## Summary to Remember

While parallelization looks like magic, keep these strict rules in mind:

1. **The Overhead Cost:** Creating new processes takes time (spawning engines, copying data to memory). For tiny tasks, parallelization can actually be *slower* than a normal loop. Only use it for heavy tasks.
2. **Processes vs. Threads:** 
    * Use **`ProcessPoolExecutor`** (Multiprocessing) for CPU-heavy tasks like math and array operations.
    * Use **`ThreadPoolExecutor`** (Multithreading) for I/O-heavy tasks like downloading websites or reading files.
3. **Amdahl's Law:** A program can never run faster than the parts of it that *must* remain sequential (like loading the initial data).